In [1]:
///- --package . --timeout 1200000
:< sm'_operators
:< document

:! requested = ()
:! reserved = ()
:! approved = ()
:! ordered = ()
:! received = ()
:! matched = ()
:! paid = ()

:== order_line = { item : string; amount : decimal::decimal }

:| private purchase s =
    | Requested :: order_line -> purchase requested
    | Reserved :: order_line -> purchase reserved
    | Approved :: order_line * list string -> purchase approved
    | Ordered :: order_line * string -> purchase ordered
    | Received :: order_line * string * decimal::decimal -> purchase received
    | Matched :: order_line * string * string -> purchase matched
    | Paid :: order_line * string * string * string -> purchase paid

:= same (x : decimal::decimal) (y : decimal::decimal) : bool = decimal::compare x y = 0i32

:= request (item : string) (amount : decimal::decimal) : purchase requested = Requested { item = item; amount = amount }

:= reserve (p : purchase requested) (budget_left : decimal::decimal) : option (purchase reserved) =
    ?= p ?|
    | Requested line => ? decimal::compare line::amount budget_left <= 0i32 ?> Some (Reserved line) |: None

:= approve (p : purchase reserved) (approvers : list string) (quorum : i32) : option (purchase approved) =
    ?= p ?|
    | Reserved line => ? listm::length approvers >= quorum ?> Some (Approved (line, approvers)) |: None

:= place_order (p : purchase approved) (order_number : string) : purchase ordered =
    ?= p ?|
    | Approved (line, _) => Ordered (line, order_number)

:= receive (p : purchase ordered) (amount : decimal::decimal) : purchase received =
    ?= p ?|
    | Ordered (line, order_number) => Received (line, order_number, amount)

:= receive_more (p : purchase received) (amount : decimal::decimal) : purchase received =
    ?= p ?|
    | Received (line, order_number, so_far) => Received (line, order_number, decimal::add so_far amount)

:= three_way (ordered : decimal::decimal) (received : decimal::decimal) (billed : decimal::decimal) : bool =
    same ordered received && same received billed

:= match_invoice \* r f {canon_of}.
    (p : purchase received) (invoice : f) (authorization : lifecycle::document r lifecycle::authorized)
    : option (purchase matched)
    =
    ?= p ?|
    | Received (line, order_number, got) =>
        ? three_way line::amount got (amount_in (canon_of invoice)::with_tax)
        ?> Some (Matched (line, order_number, lifecycle::key_text (lifecycle::key_of authorization)))
        |: None

:= pay (p : purchase matched) (reference : string) : purchase paid =
    ?= p ?|
    | Matched (line, order_number, key) => Paid (line, order_number, key, reference)

:= invoice_key_of (p : purchase paid) : string =
    ?= p ?|
    | Paid (_, _, key, _) => key

:= cents (units : i64) : decimal::decimal = { units = units; scale = 2i32 }

:= expected \* a. (what : string) (o : option a) : a =
    ?= o ?|
    | Some x => x
    | None => failwith what

:= nfe_authorization () : lifecycle::document lifecycle::nfe lifecycle::authorized =
    := issued =
        lifecycle::issue (lifecycle::nfe "nfe") (lifecycle::access_key "35261100000000000000550010000000011000000010")
    lifecycle::authorize issued (lifecycle::protocol_id "135260000000001")

:= invoice_for (net : i64) (tax : i64) : manual_invoice =
    manual_invoice ("A-1", "2026-11-03", "BRL", "Seller Ltda", "Buyer SA", cents net, cents tax)

:= approved_purchase (amount : i64) : purchase approved =
    := held = expected "the budget covers the order" (reserve (request "widget" (cents amount)) (cents 100000i64))
    expected "two of two approvers sign" (approve held [ "controller"; "cfo" ] 2i32)

:= received_purchase (ordered_amount : i64) (received_amount : i64) : purchase received =
    receive (place_order (approved_purchase ordered_amount) "PO-7") (cents received_amount)

unsupported on this machine: `zig` was not found

#### a_purchase_flows_from_requisition_to_payment (test)

In [2]:
///- --test
///> rust

:= delivered = receive_more (received_purchase 11800i64 6000i64) (cents 5800i64)
?= match_invoice delivered (invoice_for 10000i64 1800i64) (nfe_authorization ()) ?|
| Some m =>
    ? invoice_key_of (pay m "PIX-1") <> "35261100000000000000550010000000011000000010"
    ?> failwith "the paid purchase names the authorized NF-e it matched"
| None => failwith "order 118.00, receipts 60.00 + 58.00 and BT-112 118.00 match"

0

#### a_three_way_mismatch_does_not_match (test)

In [3]:
///- --test
///> rust

?= match_invoice (received_purchase 11800i64 10000i64) (invoice_for 10000i64 1800i64) (nfe_authorization ()) ?|
| Some _ => failwith "a short receipt does not match"
| None => ()
?= match_invoice (received_purchase 11800i64 11800i64) (invoice_for 10000i64 2000i64) (nfe_authorization ()) ?|
| Some _ => failwith "an invoice over the order does not match"
| None => ()

0

#### budget_and_quorum_gate_a_purchase (test)

In [4]:
///- --test
///> rust

?= reserve (request "widget" (cents 11800i64)) (cents 10000i64) ?|
| Some _ => failwith "118.00 does not fit a 100.00 budget"
| None => ()
:= held = expected "118.00 fits a 200.00 budget" (reserve (request "widget" (cents 11800i64)) (cents 20000i64))
?= approve held [ "controller" ] 2i32 ?|
| Some _ => failwith "one approver is not a quorum of two"
| None => ()

0

#### approving_an_unreserved_requisition_does_not_type (test)

In [5]:
///- --test --fails "Unification failure"
///> rust

approve (request "widget" (cents 100i64)) [ "controller" ] 1i32 |> ignore

failed to compile as expected ("Unification failure"): FatalError: File main has a type error somewhere in its path. /home/runner/work/fiscal/fiscal/src/p2p.spi:121:14 (main.spi:91:14): ETY-TermError: Unification failure. Got:      purchase requested Expected: purchase reserved In the definition: spiral_cells_test.


#### ordering_an_unapproved_requisition_does_not_type (test)

In [6]:
///- --test --fails "Unification failure"
///> rust

place_order (request "widget" (cents 100i64)) "PO-7" |> ignore

failed to compile as expected ("Unification failure"): FatalError: File main has a type error somewhere in its path. /home/runner/work/fiscal/fiscal/src/p2p.spi:125:18 (main.spi:91:18): ETY-TermError: Unification failure. Got:      purchase requested Expected: purchase approved In the definition: spiral_cells_test.


#### approving_twice_does_not_type (test)

In [7]:
///- --test --fails "Unification failure"
///> rust

approve (approved_purchase 100i64) [ "controller" ] 1i32 |> ignore

failed to compile as expected ("Unification failure"): FatalError: File main has a type error somewhere in its path. /home/runner/work/fiscal/fiscal/src/p2p.spi:129:14 (main.spi:91:14): ETY-TermError: Unification failure. Got:      purchase approved Expected: purchase reserved In the definition: spiral_cells_test.


#### receiving_before_ordering_does_not_type (test)

In [8]:
///- --test --fails "Unification failure"
///> rust

receive (approved_purchase 100i64) (cents 100i64) |> ignore

failed to compile as expected ("Unification failure"): FatalError: File main has a type error somewhere in its path. /home/runner/work/fiscal/fiscal/src/p2p.spi:133:14 (main.spi:91:14): ETY-TermError: Unification failure. Got:      purchase approved Expected: purchase ordered In the definition: spiral_cells_test.


#### matching_before_receipt_does_not_type (test)

In [9]:
///- --test --fails "Unification failure"
///> rust

match_invoice (place_order (approved_purchase 100i64) "PO-7") (invoice_for 100i64 0i64) (nfe_authorization ())
|> ignore

failed to compile as expected ("Unification failure"): FatalError: File main has a type error somewhere in its path. /home/runner/work/fiscal/fiscal/src/p2p.spi:137:20 (main.spi:91:20): ETY-TermError: Unification failure. Got:      purchase ordered Expected: purchase received In the definition: spiral_cells_test.


#### matching_an_unauthorized_invoice_does_not_type (test)

In [10]:
///- --test --fails "Unification failure"
///> rust

:= issued =
    lifecycle::issue (lifecycle::nfe "nfe") (lifecycle::access_key "35261100000000000000550010000000011000000010")
match_invoice (received_purchase 100i64 100i64) (invoice_for 100i64 0i64) issued |> ignore

failed to compile as expected ("Unification failure"): FatalError: File main has a type error somewhere in its path. /home/runner/work/fiscal/fiscal/src/p2p.spi:144:79 (main.spi:93:79): ETY-TermError: Unification failure. Got:      document _ authorized Expected: document nfe issued In the definition: spiral_cells_test.


#### matching_a_cancelled_invoice_does_not_type (test)

In [11]:
///- --test --fails "Unification failure"
///> rust

:= cancelled = lifecycle::cancel (nfe_authorization ()) (lifecycle::protocol_id "135260000000002")
match_invoice (received_purchase 100i64 100i64) (invoice_for 100i64 0i64) cancelled |> ignore

failed to compile as expected ("Unification failure"): FatalError: File main has a type error somewhere in its path. /home/runner/work/fiscal/fiscal/src/p2p.spi:149:79 (main.spi:92:79): ETY-TermError: Unification failure. Got:      document _ authorized Expected: document nfe cancelled In the definition: spiral_cells_test.


#### matching_a_string_as_the_invoice_does_not_type (test)

In [12]:
///- --test --fails "Fails to satisfy: canon_of"
///> rust

match_invoice (received_purchase 100i64 100i64) "A-1" (nfe_authorization ()) |> ignore

failed to compile as expected ("Fails to satisfy: canon_of"): FatalError: File main has a type error somewhere in its path. /home/runner/work/fiscal/fiscal/src/p2p.spi:153:53 (main.spi:91:53): ETY-ConstraintError: Constraint satisfaction error. Got: string Fails to satisfy: canon_of In the definition: spiral_cells_test.


#### paying_an_unmatched_purchase_does_not_type (test)

In [13]:
///- --test --fails "Unification failure"
///> rust

pay (received_purchase 100i64 100i64) "PIX-1" |> ignore

failed to compile as expected ("Unification failure"): FatalError: File main has a type error somewhere in its path. /home/runner/work/fiscal/fiscal/src/p2p.spi:157:10 (main.spi:91:10): ETY-TermError: Unification failure. Got:      purchase received Expected: purchase matched In the definition: spiral_cells_test.


#### paying_twice_does_not_type (test)

In [14]:
///- --test --fails "Unification failure"
///> rust

:= m =
    expected
        "100.00 matches"
        (match_invoice (received_purchase 100i64 100i64) (invoice_for 100i64 0i64) (nfe_authorization ()))
pay (pay m "PIX-1") "PIX-2" |> ignore

failed to compile as expected ("Unification failure"): FatalError: File main has a type error somewhere in its path. /home/runner/work/fiscal/fiscal/src/p2p.spi:165:10 (main.spi:95:10): ETY-TermError: Unification failure. Got:      purchase paid Expected: purchase matched In the definition: spiral_cells_test.


#### receiving_a_paid_purchase_does_not_type (test)

In [15]:
///- --test --fails "Unification failure"
///> rust

:= m =
    expected
        "100.00 matches"
        (match_invoice (received_purchase 100i64 100i64) (invoice_for 100i64 0i64) (nfe_authorization ()))
receive (pay m "PIX-1") (cents 100i64) |> ignore

failed to compile as expected ("Unification failure"): FatalError: File main has a type error somewhere in its path. /home/runner/work/fiscal/fiscal/src/p2p.spi:173:14 (main.spi:95:14): ETY-TermError: Unification failure. Got:      purchase paid Expected: purchase ordered In the definition: spiral_cells_test.
